# Lab 11 - Deep Q Learning

We are going to look at the practical implementation of a Deep Q Learning Model on the "CartPole" task from Gymnasium (https://gymnasium.farama.org/).

Our agent has to decide between two actions - moving a cart left or right - so that the pole attached to it stays balanced \& upright. You can find more information about the environment and other more challenging environments at Gymnasium’s website.

As the agent observes the current state of the environment and chooses an action, the environment transitions to a new state, and also returns a reward that indicates the consequences of the action. In this task, rewards are +1 for every incremental timestep and the environment terminates if the pole falls over too far or the cart moves more than 2.4 units away from center. This means better performing scenarios will run for longer duration, accumulating larger return.

The CartPole task is designed so that the inputs to the agent are 4 real values representing the environment state (position, velocity, etc.). We take these 4 inputs without any scaling and pass them through a small fully-connected network with 2 outputs, one for each action. The network is trained to predict the expected value for each action, given the input state. The action with the highest expected value is then chosen.

<img src='https://docs.pytorch.org/tutorials/_images/cartpole.gif' width='500px'/>

(Source: https://docs.pytorch.org/tutorials/intermediate/reinforcement_q_learning.html)


## Reminder On Deep Q Networks


Deep Q-Learning uses a neural network to approximate $Q$ functions. Hence, we usually refer to this algorithm as DQN (for *deep Q network*).

The parameters of the neural network are denoted by $\theta$.
*   As input, the network takes a state $s$,
*   As output, the network returns $Q(s, a, \theta)$, the value of each action $a$ in state $s$, according to the parameters $\theta$.


The goal of Deep Q-Learning is to learn the parameters $\theta$ so that $Q(s, a, \theta)$ approximates well the optimal $Q$-function $Q^*(s, a)$.

In addition to the network with parameters $\theta$, the algorithm keeps another network with the same architecture and parameters $\theta^-$, called **target network**.

The algorithm works as follows:

1.   At each time $t$, the agent is in state $s_t$ and has observed the transitions $(s_i, a_i, r_i, s_i')_{i=1}^{t-1}$, which are stored in a **replay buffer**.

2.  Choose action $a_t = \arg\max_a Q(s_t, a)$ with probability $1-\varepsilon_t$, and $a_t$=random action with probability $\varepsilon_t$.

3. Take action $a_t$, observe reward $r_t$ and next state $s_t'$.

4. Add transition $(s_t, a_t, r_t, s_t')$ to the **replay buffer**.

4.  Sample a minibatch $\mathcal{B}$ containing $B$ transitions from the replay buffer. Using this minibatch, we define the loss:

$$
L(\theta) = \sum_{(s_i, a_i, r_i, s_i') \in \mathcal{B}}
\left[
Q(s_i, a_i, \theta) -  y_i
\right]^2
$$
where the $y_i$ are the **targets** computed with the **target network** $\theta^-$:

$$
y_i = r_i + \gamma \max_{a'} Q(s_i', a', \theta^-).
$$

5. Update the parameters $\theta$ to minimize the loss, e.g., with gradient descent (**keeping $\theta^-$ fixed**):
$$
\theta \gets \theta - \eta \nabla_\theta L(\theta)
$$
where $\eta$ is the optimization learning rate.

6. Every $N$ transitions ($t\mod N$ = 0), update target parameters: $\theta^- \gets \theta$.

7. $t \gets t+1$. Stop if $t = T$, otherwise go to step 2.


In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
from collections import deque

import matplotlib.pyplot as plt
from matplotlib import animation
import IPython.display as ipy_display 

## Hyperparameters & Model Definintion

In [ ]:

LEARNING_RATE = 0.001
GAMMA = 0.99
MEMORY_SIZE = 10000
BATCH_SIZE = 64
EPSILON_START = 1.0
EPSILON_END = 0.01
EPSILON_DECAY = 0.995
TARGET_UPDATE = 10


Our model takes in the state vector as input, and outputs Q-scores for each of the two possible actions.

In [ ]:
class DQN(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(DQN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim)
        )

    def forward(self, x):
        return self.net(x)


## Setup Environment

In [ ]:
env = gym.make("CartPole-v1")
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

policy_net = DQN(state_dim, action_dim)
target_net = DQN(state_dim, action_dim)
target_net.load_state_dict(policy_net.state_dict())

optimizer = optim.Adam(policy_net.parameters(), lr=LEARNING_RATE)
memory = deque(maxlen=MEMORY_SIZE)
epsilon = EPSILON_START



Note the roles of `policy_net` and `target_net`.

`policy_net`: This is the "student". It is updated every single step. It decides what to do right now.

`target_net`: This is the "teacher". It is a copy of the student, but its weights are frozen. We use it to calculate what the "correct" answer should have been.

Note how training is performed on each of these

Also of note, is the structure of `env.observation_space` and `env.action_space`:

In [ ]:
env.observation_space

Refer to https://gymnasium.farama.org/environments/classic_control/cart_pole/ for a detailed explanation.

In short, there are 4 state variables:

0. Cart Position
1. Cart Velocity
2. Pole Angle
3. Pole Angular Velocity

The first array in the Box corresponds to the lower bounds of each state variable. The second array corresponds to the upper bound. The third tuple is the shape of the observations (in this case a vector of the 4 state variables), and the final element is the data type of the state variables

In [ ]:
env.action_space

## Training

In [ ]:
for episode in range(200):
    state, _ = env.reset()
    total_reward = 0
    done = False

    while not done:
        # Epsilon-greedy action selection
        if random.random() < epsilon:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                state_t = torch.FloatTensor(state).unsqueeze(0)
                action = policy_net(state_t).argmax().item()

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        memory.append((state, action, reward, next_state, done))
        state = next_state
        total_reward += reward

        # Optimization step
        if len(memory) > BATCH_SIZE:
            batch = random.sample(memory, BATCH_SIZE)
            states, actions, rewards, next_states, dones = zip(*batch)

            states = torch.FloatTensor(np.array(states))
            actions = torch.LongTensor(actions).unsqueeze(1)
            rewards = torch.FloatTensor(rewards)
            next_states = torch.FloatTensor(np.array(next_states))
            dones = torch.FloatTensor(dones)

            # Current Q-values
            current_q = policy_net(states).gather(1, actions)

            # Target Q-values using the Bellman Equation:
            # $Q(s, a) = r + \gamma \max_{a'} Q_{target}(s', a')$
            with torch.no_grad():
                max_next_q = target_net(next_states).max(1)[0]
                target_q = rewards + (1 - dones) * GAMMA * max_next_q

            loss = nn.MSELoss()(current_q.squeeze(), target_q)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # Update epsilon and target network
    epsilon = max(EPSILON_END, epsilon * EPSILON_DECAY)
    if episode % TARGET_UPDATE == 0:
        target_net.load_state_dict(policy_net.state_dict())

    if (episode + 1) % 10 == 0:
        print(f"Episode {episode+1} | Reward: {total_reward} | Epsilon: {epsilon:.2f}")

env.close()

## Visualisation

In [ ]:


def record_video(net, env_name="CartPole-v1"):
    # Using 'render_fps' helps the animation timing
    env = gym.make(env_name, render_mode="rgb_array")
    state, _ = env.reset()
    frames = []
    done = False
    
    net.eval()
    while not done:
        frames.append(env.render())
        with torch.no_grad():
            state_t = torch.FloatTensor(state).unsqueeze(0)
            action = net(state_t).argmax().item()
        state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
    
    env.close()
    return frames

def display_video(frames):
    if not frames:
        print("No frames recorded!")
        return

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.axis('off')
    patch = ax.imshow(frames[0])

    def animate(i):
        patch.set_data(frames[i])
        return [patch]

    anim = animation.FuncAnimation(
        fig, animate, frames=len(frames), interval=50, blit=True
    )
    
    # Use the alias specifically to call the display function
    # to_jshtml() is the key to avoiding ffmpeg
    ipy_display.display(ipy_display.HTML(anim.to_jshtml()))
    plt.close()

# Run the simulation and display
video_frames = record_video(policy_net)
display_video(video_frames)

### Where to go from here:


You can modify the DQN model in exactly the same manner as any lab you've seen so far, so long as you understand the correspondance between the input \& output shapes, and the state \& action spaces.